In [1]:
import numpy as np
import torch

In [2]:
class Layer:
    def __init__(self, n_in, n_out, activation='sigmoid'):
        """ Initializes a layer with given input/output sizes and activation function."""

        self.weights = torch.randn(n_out, n_in) * 0.1
        self.bias = torch.zeros(n_out) * 0.1
        self.activation_function = activation
        self.inputs = self.z = self.a = None

        self.mw = torch.zeros_like(self.weights)
        self.mb = torch.zeros_like(self.bias)
        self.vw = torch.zeros_like(self.weights)
        self.vb = torch.zeros_like(self.bias)

    def activate(self, x):
        if self.activation_function == 'sigmoid':
            return 1 / (1 + torch.exp(-x))
        if self.activation_function == 'relu':
            return torch.clamp(x, min=0)
        if self.activation_function == 'tanh':
            return torch.tanh(x)
        if self.activation_function == 'softmax':
            e = torch.exp(x - x.max(dim=-1, keepdim=True).values)
            return e / e.sum(dim=-1, keepdim=True)
        return x

    def act_deriv(self):
        if self.activation_function == 'sigmoid':
            return self.a * (1 - self.a)
        if self.activation_function == 'relu':
            return (self.z > 0).float()
        if self.activation_function == 'tanh':
            return 1 - self.a ** 2
        return torch.ones_like(self.z)

    def forward(self, inputs):
        self.inputs = inputs
        self.z = inputs @ self.weights.T + self.bias
        self.a = self.activate(self.z)
        return self.a

    def backward(self, da, lr, optimizer='adam', t=1, b1=0.9, b2=0.999, eps=1e-8):
        m = self.inputs.shape[0]
        dz = da if self.activation_function == 'softmax' else da * self.act_deriv()
        dw = (dz.T @ self.inputs) / m
        db = dz.sum(dim=0) / m
        da_prev = dz @ self.weights

        if optimizer == 'sgd':
            self.weights -= lr * dw
            self.bias -= lr * db
        elif optimizer == 'adam':
            # Momentum
            self.mw = b1 * self.mw + (1 - b1) * dw
            self.mb = b1 * self.mb + (1 - b1) * db
            
            # RMSProp
            self.vw = b2 * self.vw + (1 - b2) * dw ** 2
            self.vb = b2 * self.vb + (1 - b2) * db ** 2

            # Bias correction
            mw_hat = self.mw / (1 - b1 ** t)
            mb_hat = self.mb / (1 - b1 ** t)
            vw_hat = self.vw / (1 - b2 ** t)
            vb_hat = self.vb / (1 - b2 ** t)

            # Update parameters
            self.weights -= lr * mw_hat / (torch.sqrt(vw_hat) + eps)
            self.bias -= lr * mb_hat / (torch.sqrt(vb_hat) + eps)

        return da_prev


In [3]:
class NeuralNetwork:
    def __init__(self):
        self.layers = []

    def add_layer(self, num_inputs=None, num_neurons=None, activation='sigmoid'):
        if not self.layers:
            if num_inputs is None:
                raise ValueError("First layer must specify num_inputs.")
        else:
            num_inputs = self.layers[-1].weights.shape[0]
        self.layers.append(Layer(num_inputs, num_neurons, activation))

    def forward(self, inputs):
        """ Forward pass through the network. Handles both softmax and other activations."""
        
        for layer in self.layers:
            inputs = layer.forward(inputs)
        return inputs

    def compute_loss(self, y_pred, y_true):
        """ Compute cross-entropy loss. Handles both softmax and other activations."""
        
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        return -(y_true * torch.log(y_pred)).sum() / y_true.shape[0]

    def backward(self, y_pred, y_true, lr, optimizer='adam', t=1, b1=0.9, b2=0.999, eps=1e-8):
        """ Backpropagation through the network. Handles both softmax and other activations."""

        if self.layers[-1].activation_function == 'softmax':
            da = y_pred - y_true
        else:
            yp = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
            da = (-y_true / yp + (1 - y_true) / (1 - yp)) / y_true.shape[0]
        for layer in reversed(self.layers):
            da = layer.backward(da, lr, optimizer, t, b1, b2, eps)

    def train(self, X, y, epochs=1000, lr=0.001, optimizer='adam',
              batch_size=None, b1=0.9, b2=0.999, eps=1e-8, print_every=100):
        """ Train the neural network using either full batch or mini-batch gradient descent."""

        n = X.shape[0]
        t = 0

        for epoch in range(epochs):
            # If batch_size is None or larger than dataset, do full batch update
            if batch_size is None or batch_size >= n:
                t += 1
                y_pred = self.forward(X)
                loss = self.compute_loss(y_pred, y).item()
                self.backward(y_pred, y, lr, optimizer, t, b1, b2, eps)
            
            # Otherwise, do mini-batch updates
            else:
                loss = 0.0
                for i in range(0, n, batch_size):
                    t += 1
                    X_batch = X[i:i+batch_size]
                    y_batch = y[i:i+batch_size]
                    y_pred = self.forward(X_batch)
                    loss += self.compute_loss(y_pred, y_batch).item() * X_batch.shape[0]
                    self.backward(y_pred, y_batch, lr, optimizer, t, b1, b2, eps)
                loss /= n

            if (epoch + 1) % print_every == 0:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f}")


In [4]:
X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y = torch.tensor([[1, 0], [0, 1], [0, 1], [1, 0]], dtype=torch.float32)


for opt, lr_val, bs in [('sgd', 1.0, None), ('sgd', 1.0, 1), ('sgd', 1.0, 2), ('adam', 0.01, None)]:
    name = {'sgd': 'SGD', 'adam': 'Adam'}[opt]
    mode = 'batch' if bs is None else f'batch_size={bs}'
    print(f"\nTraining with {name} optimizer, learning rate={lr_val}, mode={mode}")

    net = NeuralNetwork()
    net.add_layer(num_inputs=2, num_neurons=4, activation='sigmoid')
    net.add_layer(num_neurons=2, activation='softmax')
    net.train(X, y, epochs=3000, lr=lr_val, optimizer=opt, batch_size=bs, print_every=1000)

    output = net.forward(X)
    preds = torch.argmax(output, dim=1)
    truth = torch.argmax(y, dim=1)

    print("\nPredictions vs True Labels:")
    for i in range(len(X)):
        print(f"  {X[i].numpy()} -> pred: {preds[i].item()}, true: {truth[i].item()}")



Training with SGD optimizer, learning rate=1.0, mode=batch
Epoch 1000/3000 - Loss: 0.6931
Epoch 2000/3000 - Loss: 0.5986
Epoch 3000/3000 - Loss: 0.0056

Predictions vs True Labels:
  [0. 0.] -> pred: 0, true: 0
  [0. 1.] -> pred: 1, true: 1
  [1. 0.] -> pred: 1, true: 1
  [1. 1.] -> pred: 0, true: 0

Training with SGD optimizer, learning rate=1.0, mode=batch_size=1
Epoch 1000/3000 - Loss: 0.0061
Epoch 2000/3000 - Loss: 0.0008
Epoch 3000/3000 - Loss: 0.0004

Predictions vs True Labels:
  [0. 0.] -> pred: 0, true: 0
  [0. 1.] -> pred: 1, true: 1
  [1. 0.] -> pred: 1, true: 1
  [1. 1.] -> pred: 0, true: 0

Training with SGD optimizer, learning rate=1.0, mode=batch_size=2
Epoch 1000/3000 - Loss: 0.6931
Epoch 2000/3000 - Loss: 0.6931
Epoch 3000/3000 - Loss: 0.6931

Predictions vs True Labels:
  [0. 0.] -> pred: 0, true: 0
  [0. 1.] -> pred: 0, true: 1
  [1. 0.] -> pred: 0, true: 1
  [1. 1.] -> pred: 0, true: 0

Training with Adam optimizer, learning rate=0.01, mode=batch
Epoch 1000/3000 - 